<a href="https://colab.research.google.com/github/vishisht-gupta/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q huggingface_hub duckdb pyarrow

from huggingface_hub import snapshot_download
from google.colab import userdata
import duckdb

In [2]:
HF_TOKEN = userdata.get("internship")
print("Token Loaded Successfully!")

Token Loaded Successfully!


In [9]:
repo_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

print(repo_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [10]:
march_file = f"{repo_path}/fact_content_daily_performance/month=2026-03/data_0.parquet"
print(march_file)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [11]:
import duckdb
con = duckdb.connect()

In [12]:
df = con.execute(f"""
SELECT *
FROM read_parquet('{march_file}')
""").df()
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [13]:
feature_df = con.execute(f"""
SELECT
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_engaged_sessions
FROM read_parquet('{march_file}')
WHERE gsc_data_available IS TRUE
""").df()

feature_df.head()

,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>


In [14]:
feature_df["ctr"] = (
    feature_df["gsc_clicks"] /
    feature_df["gsc_impressions"].replace(0, 1)
)
feature_df = feature_df.fillna(0)

feature_df.head()

,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,ctr
0,content_b7e512995f79d5a6,20,0,3.350000,0,0,0.000
1,content_05597932fe4da067,1,0,0.000000,0,0,0.000
2,content_7a105f548d9c6916,125,1,4.928000,0,0,0.008
3,content_905aa32a0230694e,7,0,4.000000,0,0,0.000
4,content_a3ea9792f793ec72,11,0,2.272727,0,0,0.000


Feature Notes

gsc_impressions
Meaning: Number of Google Search impressions.
Missing values: Filled with 0.
Available when? Yes, before prediction.

gsc_clicks
Meaning: Number of clicks from Google Search.
Missing values: Filled with 0.
Available when? Yes.

gsc_avg_position
Meaning: Average ranking position.
Missing values: Filled with 0.
Available when? Yes.

ga4_pageviews
Meaning: Number of page views.
Missing values: Filled with 0.
Available when? Yes.

ga4_engaged_sessions
Meaning: Number of engaged sessions.
Missing values: Filled with 0.
Available when? Yes.

ctr (engineered)
Meaning: Click-through rate.
Missing values: Protected from divide-by-zero.
Available when? Yes.

In [15]:
feature_df["high_clicks"] = (
    feature_df["gsc_clicks"] >
    feature_df["gsc_clicks"].median()
).astype(int)
feature_df["LEAKY_FEATURE"] = feature_df["high_clicks"]
feature_df[["high_clicks", "LEAKY_FEATURE"]].head()

,high_clicks,LEAKY_FEATURE
0,0,0
1,0,0
2,1,1
3,0,0
4,0,0


In [16]:
feature_df.drop(columns=["LEAKY_FEATURE"], inplace=True)
feature_df.head()

,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,ctr,high_clicks
0,content_b7e512995f79d5a6,20,0,3.350000,0,0,0.000,0
1,content_05597932fe4da067,1,0,0.000000,0,0,0.000,0
2,content_7a105f548d9c6916,125,1,4.928000,0,0,0.008,1
3,content_905aa32a0230694e,7,0,4.000000,0,0,0.000,0
4,content_a3ea9792f793ec72,11,0,2.272727,0,0,0.000,0


Leakage Check

I intentionally created `LEAKY_FEATURE` directly from the target (`high_clicks`).

This feature contains the answer and would produce unrealistically high model performance.

It was removed before any modeling to avoid data leakage.

Excluded Fields

- report_date → Time identifier, not used as a feature.
- client_hash_id → Unique identifier.
- content_hash_id → Unique identifier.
- client_has_gsc → Configuration flag.
- client_has_ga4 → Configuration flag.
- gsc_data_available → Availability flag.
- ga4_data_available → Availability flag.
- month → Partition column, not a predictive feature.

Feature vector created
Missing values handled
Engineered feature added
Leakage demonstrated
Leaky feature remove
Excluded fields documented
Notebook runs successfully from top to bottom